<a href="https://colab.research.google.com/github/padmarajanandhu-code/Case_study_sql/blob/main/data_Acquisition_Casestudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import requests
import pandas as pd

launch_url = "https://api.spacexdata.com/v4/launches"
launch_data = requests.get(launch_url).json()

launch_df = pd.DataFrame(launch_data)

launch_df = launch_df[['name', 'date_utc', 'success', 'details', 'rocket']]

launch_df['date_utc'] = pd.to_datetime(launch_df['date_utc'])
launch_df['year'] = launch_df['date_utc'].dt.year

launch_df.head()

,name,date_utc,success,details,rocket,year
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009


In [2]:
rocket_url = "https://api.spacexdata.com/v4/rockets"
rocket_data = requests.get(rocket_url).json()

rocket_df = pd.DataFrame(rocket_data)

rocket_df = rocket_df[['id', 'name', 'type', 'active', 'stages']]

rocket_df.head()

,id,name,type,active,stages
0,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2
2,5e9d0d95eda69974db09d1ed,Falcon Heavy,rocket,True,2
3,5e9d0d96eda699382d09d1ee,Starship,rocket,False,2


In [3]:
# Merge on rocket ID
merged_df = launch_df.merge(rocket_df, left_on='rocket', right_on='id', how='left')

merged_df.head()

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2


In [4]:
import numpy as np

countries = ['USA', 'Russia', 'India', 'China', 'France']
merged_df['country'] = np.random.choice(countries, size=len(merged_df))

merged_df.head()

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages,country
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,China
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,USA
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,India
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,USA
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,Russia


In [5]:
import sqlite3

conn = sqlite3.connect("spacex_launches.db")

merged_df.to_sql("launches", conn, if_exists="replace", index=False)

print("Data stored successfully!")

Data stored successfully!


In [6]:
query1 = """
SELECT country, COUNT(*) as total_launches
FROM launches
GROUP BY country
ORDER BY total_launches DESC;
"""

pd.read_sql_query(query1, conn)

,country,total_launches
0,France,45
1,India,44
2,Russia,43
3,China,38
4,USA,35


In [7]:
query2 = """
SELECT year, COUNT(*) as total_launches
FROM launches
GROUP BY year
ORDER BY total_launches DESC
LIMIT 1;
"""

pd.read_sql_query(query2, conn)

,year,total_launches
0,2022,62


In [10]:
query3 = """
SELECT name_x, COUNT(*) as launch_count
FROM launches
GROUP BY name_x
ORDER BY launch_count DESC
LIMIT 5;
"""

pd.read_sql_query(query3, conn)

,name_x,launch_count
0,ispace Mission 1 & Rashid,1
1,ZUMA,1
2,WorldView Legion 1 & 2,1
3,Viasat-3 & Arcturus,1
4,USSF-44,1
